# AI301 Assignment #2 — EfficientNet-B4 on the cleaned data, scene-grouped validation

Same model, training loop and augmentations as `assignment-2-dl-sample-submission.ipynb`, with three changes:
1. **Data:** `train_clean.csv` from the EDA notebook (label fixes, train–train duplicates removed, `scene_group`, `is_val`). No cleaning here.
2. **Validation:** the fixed **scene-grouped 20% hold-out** (`is_val`): rare pets appear in val only in scenes the model never saw in training,
   so val measures recognising the pet, not the room. One model per run (the val images are "fold 0").
3. **After the run:** thresholds tuned rare classes first, then the species rule, then Cat / Dog (middle of the best plateau),
   flip-TTA for the final val and test predictions, sanity checks on the submission and a results log.

```
train_clean.csv (is_val fixed in the EDA)
   ├── is_val = False ─► TRAIN: real images as they are + RARE_COPIES synthetic rare-pet images per paste source per epoch
   │                      (pet cut out with a CAM box, pasted on a random label-less train photo, flip + colour jitter)
   └── is_val = True  ─► VAL: original images, never augmented; rare pets only in unseen scenes
per epoch:  train ─► predict val ─► val mAP ─► keep best checkpoint (early stopping on val mAP)
after run:  best checkpoint ─► flip-TTA val probs ─► thresholds ─► species rule ─► test ─► sanity checks ─► submission
```

**Kaggle setup:** Accelerator **GPU T4 / P100**, **Internet on** (timm weights, W&B), inputs: the competition data and `train_clean.csv`
(upload it as a dataset, or add the EDA notebook's output). W&B: Add-ons → Secrets → `WANDB_API_KEY` of the account you log to.

---
**NEW (AI-assisted, Claude): this is `eff_net_augumented.ipynb` plus the additions below (that file itself is unchanged).** With `PRESET = "b4_gap"` it reproduces the EfficientNet-B4 run exactly. Every new or changed piece is marked `NEW (AI-assisted, Claude)`; everything else is unchanged.
1. **Experiment presets** (cell "Setup & config"): the same notebook now trains EfficientNet-B4 (the original run) **or ConvNeXt-Tiny**, with the usual **global-average-pooling head** or the **custom class-specific attention-pooling head**. Optimizer (AdamW, 2 LR groups, OneCycle), loss (plain BCE), augmentations (rare-pet paste only), split, early stopping and thresholds are identical for every preset, so the runs are comparable.
2. **`ClassAttentionPool`** (custom layer) + a self-check, before the model cell.
3. `stage_of` understands ConvNeXt stages (gradient-flow plot); the results log has a `head` column; an attention-map gallery for the custom head.
4. The soft-voting ensemble lives in its own notebook, `ensemble_soft_voting.ipynb`, and reads the `val_probs_*.npy` / `test_probs_*.npy` that every run saves.

## 1. Setup & config

In [ ]:
import hashlib, os, random, subprocess, time, warnings, glob

# Pick the GPU before torch starts CUDA. Locally (shared machine): use GPU 4 if it has MIN_FREE_GB free, else the freest GPU.
# Kaggle gives its own GPU(s), so leave the choice to Kaggle there.
PREFERRED_GPU, MIN_FREE_GB = 4, 30
if not os.path.exists("/kaggle/input") and "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    smi = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True).stdout
    free_gb = {int(i): int(mib) / 1024 for i, mib in (line.split(",") for line in smi.strip().splitlines())}
    gpu = PREFERRED_GPU if free_gb.get(PREFERRED_GPU, 0) >= MIN_FREE_GB else max(free_gb, key=free_gb.get)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    print(f"GPU {gpu}: {free_gb[gpu]:.0f} GB free")

# Two GPUs (Kaggle T4 x2): DataParallel copies the weights GPU0 -> GPU1 every step through NCCL. The two T4s have no direct
# link, and NCCL's peer-to-peer route over PCIe inside the VM can hang or corrupt memory. Route it through host memory
# instead (barely slower at this size). Must be set before torch starts NCCL.
os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_IB_DISABLE", "1")
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score, average_precision_score, precision_recall_curve, precision_score, recall_score
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import timm
import wandb

warnings.filterwarnings("ignore")
ON_KAGGLE = os.path.exists("/kaggle/input")

### Plot helpers
`label_code`, `label_names` and `show_grid` (inlined, so the notebook needs no local package).

In [ ]:
def label_code(r, labels):
    """0/1 string of a row's labels in `labels` order, e.g. '10010'."""
    return "".join(str(int(r[l])) for l in labels)

def label_names(r, labels):
    """Comma-separated positive labels of a row, or 'none'."""
    return ", ".join(l for l in labels if r[l] == 1) or "none"

def pet_labels(r):
    """Labels as text, species first: 'Cat: Moris, Motya | Dog: Biatrix', 'Cat' (no named cat), or 'none'."""
    parts = []
    for species, names in [("Cat", ["Moris", "Motya"]), ("Dog", ["Biatrix"])]:
        named = [name for name in names if r[name] == 1]
        if r[species] == 1 or named:
            parts.append(f"{species}: {', '.join(named)}" if named else species)
    return " | ".join(parts) or "none"

def show_grid(rows, labels=None, title=None, suptitle=None, ncols=None, size=2.6):
    """One subplot row per DataFrame in `rows`, one image (column `path`) per cell.
    `title(row)` sets each image's title; by default the label names and code (if `labels` is given) or the image_id."""
    if title is None:
        title = (lambda r: f"{label_names(r, labels)} | {label_code(r, labels)}") if labels else (lambda r: str(r.image_id)[:8])
    rows = [df for df in rows if len(df)]
    ncols = ncols or max(len(df) for df in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(size * ncols, size * len(rows)), squeeze=False)
    for ax_row, df in zip(axes, rows):
        for ax in ax_row:
            ax.axis("off")
        for ax, (_, r) in zip(ax_row, df.iterrows()):
            with Image.open(r.path) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(title(r), fontsize=7)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

In [ ]:
SEED = 42
def seed_everything(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
seed_everything()

In [ ]:
CFG = dict(
    model_name="efficientnet_b4.ra2_in1k",   # timm EfficientNet-B4, ImageNet-1k weights
    img_size=384,            # B4 was trained at 320 px and evaluated at 384 (timm pretrained_cfg); images are 450x375
    batch_size="auto",       # "auto" = from GPU memory (64 for >=30 GB, 48 for >=20 GB, 32 for a 16 GB T4 / P100, else 16);
                             # on an out-of-memory error training restarts with a 25% smaller batch
    pred_batch_size="auto",  # images per forward pass when predicting val / test (no gradients, so larger than batch_size)
    epochs=25,               # max epochs; early stopping usually ends training sooner
    monitor="val_mAP",       # metric for the best checkpoint and early stopping. val_mAP: threshold-free, every class 1/5 like macro F1,
                             # and smooth (macro F1 on 8-11 val positives per rare class jumps by single images).
                             # Alternative: "val_macroF1_tuned" (thresholds tuned on val each epoch + species rule; ties go to the better mAP)
    patience=10,             # stop after this many epochs without an improvement of `monitor` larger than min_delta
    min_delta=0.001,         # `monitor` must rise by more than this to reset the patience (any rise still saves a new best checkpoint)
    lr=1e-3,                 # peak LR of the new classifier head
    backbone_lr_mult=0.1,    # the pretrained body learns 10x slower than the head
    weight_decay=1e-2,
    drop_rate=0.3,           # dropout before the classifier
    drop_path_rate=0.2,      # stochastic depth: randomly skips MBConv blocks during training
    aug_common=True,         # albumentations for every real train image (flip, mild crop/shift, colour, grayscale, blur);
                             # False = real images as they are (the first run, Public LB 0.9612)
    tta=True,                # final val + test predictions: mean of the image and its horizontal flip (thresholds are tuned on the same)
    num_workers=min(16, os.cpu_count()),   # DataLoader workers that run the augmentations (Kaggle has 4 CPUs)
    loader_timeout=600,      # seconds to wait for one batch from the workers; a stalled worker then raises an error instead of hanging forever
    multi_gpu=False,         # False = one GPU (default). True = DataParallel over all GPUs: measured on Kaggle T4 x2 it ran at
                             # ~6 img/s and the main process grew ~110 MB of RAM per step until the machine froze
                             # (the two T4s have no direct link, every step copies batch and weights between them)
    log_every=50,            # heartbeat: print progress (and free RAM) every N training steps, so a stall is visible within minutes
    ram_reserve_gb=3.0,      # host RAM guard: below this much free RAM, free memory and restart the data workers;
                             # if that does not help, stop training early (best checkpoint is used) instead of freezing the machine
    grad_log_every=25,       # record per-stage gradient norms every N steps (for the gradient-flow plot)
    wandb_mode="online",     # "online", "offline" (sync later with `wandb sync`) or "disabled"
    wandb_project="cats-dogs-plus",
    wandb_entity=None,       # W&B user or team to log to; None = the default of the account whose API key is used
    run_tag="b4_sceneval_aug",   # short name of this experiment: change it for every run   # short name of this experiment (W&B run name, output files, results log)
    debug=False,             # True = smoke test: 640 train / 256 val images, 1 epoch
)

# ===== NEW (AI-assisted, Claude): experiment presets =====
# The CFG above is the EfficientNet-B4 setup of eff_net_augumented.ipynb and stays as it is (albumentations on, monitor = val mAP, patience 10). Pick ONE preset per run: it overrides only the
# backbone, the pooling head and the run name. Optimizer, loss, LR schedule, augmentations, split and early stopping
# stay exactly as above, so the runs differ only in what we want to compare.
#   head="gap":  the timm head (global average pooling + linear)  |  head="attn": custom class-specific attention pooling
PRESETS = {
    "b4_gap":          dict(model_name="efficientnet_b4.ra2_in1k",       head="gap",  run_tag="b4_sceneval_aug"),   # = eff_net_augumented.ipynb as it is
    "b4_attn":         dict(model_name="efficientnet_b4.ra2_in1k",       head="attn", run_tag="b4_attn_aug"),
    "convnext_t_gap":  dict(model_name="convnext_tiny.fb_in22k_ft_in1k", head="gap",  run_tag="convnext_t_aug"),
    "convnext_t_attn": dict(model_name="convnext_tiny.fb_in22k_ft_in1k", head="attn", run_tag="convnext_t_attn_aug"),
}
PRESET = "convnext_t_gap"   # <- change only this line between runs
CFG.update(PRESETS[PRESET])
try:   # fail fast on a typo in the weights tag (timm raises for a wrong tag of a known model, returns None for an unknown model)
    _known = timm.get_pretrained_cfg(CFG["model_name"]) is not None
except RuntimeError:
    _known = False
if not _known:
    raise ValueError(f"unknown timm weights {CFG['model_name']!r}; options: {timm.list_pretrained(CFG['model_name'].split('.')[0] + '*')}")
print("preset:", PRESET, "->", {k: CFG[k] for k in ("model_name", "head", "run_tag")})
# ===== END NEW =====

if CFG["debug"]:
    CFG["epochs"] = 1

RUN_NAME = CFG["run_tag"] + ("_debug" if CFG["debug"] else "")
FOLDS = [0]   # one model: the val images (is_val) are "fold 0", the train images "fold -1"

In [ ]:
# W&B: live training curves in the browser. Logs metrics only (no images), so no competition data leaves the machine.
# Kaggle: Add-ons -> Secrets -> add WANDB_API_KEY (of the account you want to log to) and attach it; Settings -> Internet on.
# Locally: run `wandb login` once, or set WANDB_API_KEY.
# Never prompt for input: in "Save & Run All" nobody can answer, so a prompt would block or repeat. Without a key -> offline.
if ON_KAGGLE and not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY").strip()
    except Exception as e:
        print("No WANDB_API_KEY secret attached to this notebook:", type(e).__name__)

if CFG["wandb_mode"] == "online":
    key = os.environ.get("WANDB_API_KEY", "")
    try:
        if not key and not ON_KAGGLE:
            ok = wandb.login(timeout=30)              # local: key stored by `wandb login`
        else:
            ok = bool(key) and wandb.login(key=key, relogin=True, timeout=30)   # explicit key: no prompt
    except Exception as e:
        print("W&B login failed:", e)
        ok = False
    if ok:
        print("W&B user:", wandb.Api().default_entity)
    else:
        print("-> W&B logs OFFLINE for this run (sync later with `wandb sync`, or attach the WANDB_API_KEY secret)")
        CFG["wandb_mode"] = "offline"
os.environ["WANDB_MODE"] = CFG["wandb_mode"]   # wandb.init never asks interactively after this
print("W&B mode:", CFG["wandb_mode"])

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| timm", timm.__version__)

if DEVICE == "cpu":
    print("\n" + "!" * 70 + "\n!! NO GPU DETECTED -> training will take hours. Enable it in\n"
          "!! Settings (right sidebar) -> Accelerator -> GPU T4 x2 / P100, then restart.\n" + "!" * 70)
else:
    print("GPU:", torch.cuda.get_device_name(0))

# Kaggle "GPU T4 x2": train on both GPUs with nn.DataParallel (each batch is split across the GPUs)
N_GPUS = (torch.cuda.device_count() if CFG["multi_gpu"] else 1) if DEVICE == "cuda" else 0
print("GPUs used for training:", max(N_GPUS, 1) if DEVICE == "cuda" else 0)
# channels_last memory layout: faster on one GPU, but with DataParallel it triggers "CUDA error: misaligned address"
# in the depthwise convolutions (T4 + AMP), so it is only used when training on a single GPU
CHANNELS_LAST = N_GPUS <= 1
MEMORY_FORMAT = torch.channels_last if CHANNELS_LAST else torch.contiguous_format

# Batch sizes from the memory of one GPU: B4 at 384 px with AMP needs roughly 0.3 GB per training image
def auto_batch_sizes():
    if DEVICE != "cuda":
        return 16, 64
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    for min_gb, train_bs, pred_bs in [(30, 64, 256), (20, 48, 192), (14, 32, 128)]:
        if gb >= min_gb:
            return train_bs, pred_bs
    return 16, 64

auto_train_bs, auto_pred_bs = (b * max(N_GPUS, 1) for b in auto_batch_sizes())   # per-GPU size x number of GPUs
if CFG["batch_size"] == "auto":
    CFG["batch_size"] = auto_train_bs
if CFG["pred_batch_size"] == "auto":
    CFG["pred_batch_size"] = auto_pred_bs
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
print(f"batch size: {CFG['batch_size']} (train) / {CFG['pred_batch_size']} (predict) | GPU memory: {gpu_gb:.1f} GB")

## 2. Data: competition images + `train_clean.csv`

In [ ]:
def find_file(name, roots):
    """First `name` under `roots`; follows symlinks (Kaggle may mount inputs through them)."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if not hits:
            hits = [os.path.join(r, name) for r, _, files in os.walk(root, followlinks=True) if name in files]
        if hits:
            return Path(sorted(hits)[0])
    return None

def show_tree(root, depth=4):
    """Print the folder structure under root (follows symlinks), to see where the inputs really are."""
    for r, dirs, files in os.walk(root, followlinks=True):
        level = r[len(str(root)):].count(os.sep)
        print("  " * level + (os.path.basename(r) or str(r)) + f"/  [{len(files)} files{': ' + ', '.join(sorted(files)[:4]) if files else ''}]")
        if level >= depth - 1:
            dirs[:] = []

if ON_KAGGLE:
    # In "Save & Run All" Kaggle can attach the inputs a little after the kernel starts (the folder exists but is still
    # empty), so look again for up to INPUT_WAIT_S seconds before giving up
    INPUT_WAIT_S = 240
    KNOWN_CSV = Path("/kaggle/input/competitions/cats-and-dogs/train.csv")
    t0 = time.time()
    while True:
        train_csv = KNOWN_CSV if KNOWN_CSV.exists() else find_file("train.csv", ["/kaggle/input"])
        # search the small folders first: a recursive search through the competition's ~31k images is slow on Kaggle
        clean_csv = find_file("train_clean.csv", ["/kaggle/input/datasets", "/kaggle/input/notebooks", "/kaggle/working", "/kaggle/input"])
        if (train_csv and clean_csv) or time.time() - t0 > INPUT_WAIT_S:
            break
        print(f"waiting for the inputs ({time.time() - t0:.0f}s): train.csv {'found' if train_csv else 'missing'}, "
              f"train_clean.csv {'found' if clean_csv else 'missing'}")
        time.sleep(15)
    CACHE_DIR, OUT_DIR = Path("/kaggle/temp"), Path("/kaggle/working")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # the kernel may start in notebooks/
    train_csv = find_file("train.csv", [ROOT / "data", ROOT])
    clean_csv = find_file("train_clean.csv", [ROOT, ROOT / "data"])
    CACHE_DIR, OUT_DIR = ROOT / "cache", ROOT / "output"

if not (train_csv and clean_csv) and os.path.isdir("/kaggle/input"):
    print("Inputs not found. Folder structure of /kaggle/input:")
    show_tree("/kaggle/input")
assert train_csv, "competition train.csv not found: attach the competition data"
assert clean_csv, "train_clean.csv not found: upload it as a dataset (or add the EDA notebook output) and attach it"
DATA_DIR = train_csv.parent
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| train_clean.csv:", clean_csv)

In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv", dtype={"image_id": str})
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"image_id": str})
LABELS = sample_sub.columns.drop("image_id").tolist()

# Cleaned train table from the EDA: fixed labels, no train-train duplicates, scene groups and the fixed validation split
train = pd.read_csv(clean_csv, dtype={"image_id": str})
train["is_val"] = train.is_val.astype(str).str.lower().eq("true") if train.is_val.dtype == object else train.is_val.astype(bool)
assert set(LABELS) <= set(train.columns) and {"scene_group", "is_val"} <= set(train.columns), "old train_clean.csv: rerun the EDA"
assert train.groupby("scene_group").is_val.nunique().max() == 1, "a scene is split across train and val"
assert set(train.image_id) <= set(pd.read_csv(train_csv, dtype={"image_id": str}).image_id), "train_clean.csv has ids that are not in the competition train.csv"

# Image paths: {DATA_DIR}/{train|test}/{image_id}.jpg (one listing per folder, fast). Only if that layout does not match,
# fall back to scanning the whole data folder (slow on Kaggle: ~31k files)
def paths_from_folder(df, split):
    folder = DATA_DIR / split
    names = set(os.listdir(folder)) if folder.is_dir() else set()
    return [str(folder / f"{i}.jpg") if f"{i}.jpg" in names else None for i in df.image_id]

train["path"], test["path"] = paths_from_folder(train, "train"), paths_from_folder(test, "test")
if train.path.isna().any() or test.path.isna().any():
    print("images not in train/ and test/ as <id>.jpg -> scanning the data folder")
    IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    id2path = {}
    for root, _, files in os.walk(DATA_DIR, followlinks=True):
        for f in files:
            if Path(f).suffix.lower() in IMG_EXT:
                id2path[Path(f).stem] = os.path.join(root, f)
    train["path"] = train.image_id.map(id2path)
    test["path"] = test.image_id.map(id2path)
assert train.path.notna().all() and test.path.notna().all(), "some images are missing"

train["fold"] = np.where(train.is_val, 0, -1)   # fold 0 = val, -1 = train only
print("labels:", LABELS)
print("train_clean", train.shape, "| test", test.shape)

In [ ]:
# The split as it comes from the EDA: every class ~20% in val, rare pets only in scenes that are not in train
none = (train[LABELS].sum(axis=1) == 0).astype(int)
split = train.assign(none=none).groupby("is_val")[LABELS + ["none"]].sum()
split.index = split.index.map({False: "train", True: "val"})
split.insert(0, "images", train.is_val.value_counts().sort_index().values)
split.loc["val_%"] = (split.loc["val"] / split.sum() * 100).round(1)
display(split)
rare_scenes = train[train.scene_group.str.startswith("scene_")]
print(f"rare-class scenes: {rare_scenes.scene_group.nunique()} | in val: {rare_scenes[rare_scenes.is_val].scene_group.nunique()}")

## 3. Images in memory, normalisation

In [ ]:
S = CFG["img_size"]

def _load(p):
    with Image.open(p) as im:
        return np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR), dtype=np.uint8)

CACHE_VERSION = "rgb_bilinear_v1"   # bump when _load changes, so old caches are not reused

# The decoded images (~5.4 GB train + ~8.3 GB test at 384 px) would not fit next to training in Kaggle's 30 GB of RAM.
# So they live in .npy files on DISK and are memory-mapped: the arrays behave as usual, but only the parts that are read
# are in RAM, and the OS can drop them again under memory pressure. A cache dir in RAM (tmpfs) would defeat that, so avoid it.
def fs_type(path):
    """File system type of `path` ("tmpfs" = lives in RAM), from /proc/mounts; "?" if unknown (e.g. macOS)."""
    if not os.path.exists("/proc/mounts"):
        return "?"
    path, best, kind = os.path.realpath(path), "", "?"
    for line in open("/proc/mounts"):
        _, mnt, t = line.split()[:3]
        if (path == mnt or path.startswith(mnt.rstrip("/") + "/")) and len(mnt) >= len(best):
            best, kind = mnt, t
    return kind

import shutil
need_gb = (len(train) + len(test)) * S * S * 3 / 1e9
for cand in [CACHE_DIR, OUT_DIR / "image_cache", Path("/tmp/image_cache")]:
    cand.mkdir(parents=True, exist_ok=True)
    if fs_type(cand) != "tmpfs" and shutil.disk_usage(cand).free / 1e9 > need_gb + 2:
        CACHE_DIR = cand
        break
print(f"image cache: {CACHE_DIR} ({fs_type(CACHE_DIR)}, {shutil.disk_usage(CACHE_DIR).free / 1e9:.0f} GB free, needs {need_gb:.1f} GB)")
if str(CACHE_DIR).startswith(str(OUT_DIR)):
    print("note: the cache is in the output folder; delete it before saving a version (shutil.rmtree(CACHE_DIR))")

def load_all(df, name):
    """uint8 NHWC images of df, memory-mapped from a .npy cache on disk (decoded once, straight into the file)."""
    paths = list(df.path)
    # Cache key covers which images, in which order, at which size and with which preprocessing:
    # a cache built for another row order or image set would silently pair images with the wrong labels
    key = hashlib.md5(("\n".join(df.image_id) + f"|{S}|{CACHE_VERSION}").encode()).hexdigest()[:12]
    f = CACHE_DIR / f"{name}_{len(paths)}_{S}_{key}.npy"
    if not f.exists():
        t0 = time.time()
        tmp = f.with_suffix(".tmp.npy")   # renamed when complete, so an interrupted run never leaves a broken cache
        arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(len(paths), S, S, 3))
        with Pool(os.cpu_count()) as pool:
            for i, a in enumerate(pool.imap(_load, paths, chunksize=64)):
                arr[i] = a
        arr.flush()
        del arr
        os.replace(tmp, f)
        print(f"{name}: {len(paths)} images decoded in {time.time() - t0:.0f}s")
    # mode "c" = copy-on-write: reads come from the file, the array counts as writable (no torch warnings), nothing is written back
    return np.load(f, mmap_mode="c")

X_train_all = load_all(train, "train")      # NHWC uint8, same row order as `train`
X_test = load_all(test, "test")             # same row order as test.csv / sample_submission

def drop_file_cache():
    """Ask the kernel to drop the cached pages of the image files (clean pages; they are re-read from disk when needed)."""
    if not hasattr(os, "posix_fadvise"):   # Linux only
        return
    for arr in (X_train_all, X_test):
        fd = os.open(arr.filename, os.O_RDONLY)
        try:
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
        finally:
            os.close(fd)

drop_file_cache()   # decoding just wrote ~14 GB through the page cache: release it before training starts
print("train", X_train_all.shape, f"{X_train_all.nbytes / 1e9:.1f} GB | test", X_test.shape, f"{X_test.nbytes / 1e9:.1f} GB (on disk, memory-mapped)")

Y_train_all = train[LABELS].values               # 0/1 labels, same row order as X_train_all

In [ ]:
# Normalise with the statistics the backbone was pretrained with (timm stores them per model)
PRETRAINED_CFG = timm.get_pretrained_cfg(CFG["model_name"])
print("pretrained input size:", PRETRAINED_CFG.input_size, "| mean:", PRETRAINED_CFG.mean, "| std:", PRETRAINED_CFG.std)

MEAN_T = torch.tensor(PRETRAINED_CFG.mean, device=DEVICE).view(1, 3, 1, 1)
STD_T = torch.tensor(PRETRAINED_CFG.std, device=DEVICE).view(1, 3, 1, 1)

def prep(xb):
    """uint8 NHWC (any device) -> float NCHW in [0,1] on DEVICE"""
    return torch.as_tensor(xb).to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)

def normalize(x):
    return ((x - MEAN_T) / STD_T).contiguous(memory_format=MEMORY_FORMAT)

### Rare-class pet boxes (CAM from the ImageNet head)
Photos of Moris, Motya and Biatrix keep coming from the same rooms, so a model can learn "this sofa" instead of "this cat".
To break that link we cut each rare pet out and paste it onto other backgrounds (next subsection). The box comes from a
**Class Activation Map** of the ImageNet-pretrained EfficientNet-B4 from `timm`, with no training on our data and no hand labels.

EfficientNet ends with global average pooling and one linear layer, so the logit of ImageNet class $c$ is the mean over
positions of $M_c(x,y)=\sum_k w_{c,k}A_k(x,y)$, where $A$ is the last feature map ($1792\times12\times12$ at 384 px).
$M_c$ shows where the evidence for class $c$ sits. ImageNet has 5 domestic cat classes (281–285) and 118 dog breeds (151–268),
so averaging $M_c$ over them, weighted by each class's probability, gives a "where is the cat" or "where is the dog" map.
The top-right cell of this network's last feature map is an artefact (huge activations on every image), so it is replaced by its neighbours. We keep the largest blob above
`CAM_THRESHOLD` of the maximum and grow its box by `BOX_MARGIN`, because CAM lights up the face more than the body.

A photo is used as a paste source (`paste_ok`) only if its labels are exactly one rare pet plus its species, so the cut-out
cannot carry a second labelled animal, and the ImageNet model is reasonably sure it sees that species (`CAM_MIN_PROB`).

In [ ]:
from scipy import ndimage

RARE = ["Moris", "Motya", "Biatrix"]
SPECIES = {"Moris": "Cat", "Motya": "Cat", "Biatrix": "Dog"}
IMAGENET_IDX = {"Cat": list(range(281, 286)), "Dog": list(range(151, 269))}   # domestic cats, dog breeds
CAM_MODEL = "efficientnet_b4.ra2_in1k"   # GAP + one linear head, so CAM is exact; ImageNet-1k weights from timm
CAM_THRESHOLD = 0.2    # share of the CAM maximum that counts as "animal"
BOX_MARGIN = 0.25      # grow the box by this share of its width/height on each side
CAM_MIN_PROB = 0.05    # ImageNet probability of the species (summed over its classes) needed to use the box;
                       # low on purpose: dark fluffy cats in cluttered rooms often get 0.05-0.2 although the CAM sits on the cat

cam_model = timm.create_model(CAM_MODEL, pretrained=True).eval().to(DEVICE)
cam_cfg = timm.get_pretrained_cfg(CAM_MODEL)
cam_mean = torch.tensor(cam_cfg.mean, device=DEVICE).view(1, 3, 1, 1)
cam_std = torch.tensor(cam_cfg.std, device=DEVICE).view(1, 3, 1, 1)
W_in1k = cam_model.get_classifier().weight   # (1000, C)

@torch.no_grad()
def species_cam(xb, species):
    """uint8 NHWC images -> CAM of the species (B, S, S) scaled to [0, 1], and the ImageNet probability of the species."""
    x = (prep(xb) - cam_mean) / cam_std
    feats = cam_model.forward_features(x)                    # (B, C, h, w), before pooling
    idx = IMAGENET_IDX[species]
    probs = cam_model.forward_head(feats).softmax(dim=1)[:, idx]   # (B, n classes of the species)
    # CAMs of the species' classes, weighted by their probability: the predicted breed dominates. A plain sum over
    # 118 dog breeds adds up many weak, noisy maps and often peaks on grass or snow
    cam = torch.einsum("bchw,nc,bn->bhw", feats, W_in1k[idx], probs)
    # The top-right cell of this network's last feature map is an artefact: on every image (cat or not) its activations
    # are 16-55x the median cell, so it would always be the CAM peak. Replace it with the mean of its 3 neighbours
    cam[:, 0, -1] = (cam[:, 0, -2] + cam[:, 1, -1] + cam[:, 1, -2]) / 3
    cam = F.interpolate(cam[:, None], size=x.shape[-2:], mode="bilinear")[:, 0].clamp(min=0)
    cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp(min=1e-6)
    return cam.cpu().numpy(), probs.sum(dim=1).cpu().numpy()

def cam_box(cam):
    """(x0, y0, x1, y1) as fractions of the image: box of the largest blob above CAM_THRESHOLD, plus BOX_MARGIN."""
    blobs, n = ndimage.label(cam >= CAM_THRESHOLD)
    if n == 0:   # empty CAM: fall back to the whole image
        return 0.0, 0.0, 1.0, 1.0
    sizes = ndimage.sum_labels(np.ones_like(cam), blobs, index=range(1, n + 1))
    ys, xs = np.nonzero(blobs == 1 + np.argmax(sizes))
    h, w = cam.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    mx, my = BOX_MARGIN * (x1 - x0), BOX_MARGIN * (y1 - y0)
    return max(0.0, (x0 - mx) / w), max(0.0, (y0 - my) / h), min(1.0, (x1 + mx) / w), min(1.0, (y1 + my) / h)

# Every train image with a rare label; its species picks the CAM (Moris/Motya -> cat map, Biatrix -> dog map)
rare_rows = np.where(train[RARE].sum(axis=1) > 0)[0]
records, rare_cams = [], {}
for r in rare_rows:
    pets = [lab for lab in RARE if train[lab].iat[r] == 1]
    species = SPECIES[pets[0]]
    cam, prob = species_cam(X_train_all[r:r + 1], species)
    rare_cams[r] = cam[0].astype(np.float16)
    labels_ok = len(pets) == 1 and {lab for lab in LABELS if train[lab].iat[r] == 1} == {species, pets[0]}
    records.append(dict(row=r, image_id=train.image_id.iat[r], rare=pets[0] if len(pets) == 1 else "+".join(pets),
                        species=species, prob=prob[0], labels_ok=labels_ok, **dict(zip(["x0", "y0", "x1", "y1"], cam_box(cam[0])))))
rare_boxes = pd.DataFrame(records)
rare_boxes["paste_ok"] = rare_boxes.labels_ok & (rare_boxes.prob >= CAM_MIN_PROB)
rare_boxes["box_area"] = (rare_boxes.x1 - rare_boxes.x0) * (rare_boxes.y1 - rare_boxes.y0)

del cam_model, W_in1k
torch.cuda.empty_cache()

summary = pd.DataFrame({
    lab: {
        "photos": int(train[lab].sum()),
        "other labels too": int((rare_boxes.rare.str.contains(lab) & ~rare_boxes.labels_ok).sum()),
        f"species prob < {CAM_MIN_PROB}": int((rare_boxes.rare.str.contains(lab) & rare_boxes.labels_ok & ~rare_boxes.paste_ok).sum()),
        "paste sources": int(((rare_boxes.rare == lab) & rare_boxes.paste_ok).sum()),
        "median box area": round(float(rare_boxes[rare_boxes.rare.str.contains(lab)].box_area.median()), 2),
    } for lab in RARE
}).T
display(summary)

In [ ]:
# Every rare-class photo with its CAM (heat) and box: green = used as a paste source, red = not used (title says why)
from matplotlib.patches import Rectangle

def show_rare_boxes(lab, ncols=8, size=2.4):
    df = rare_boxes[rare_boxes.rare.str.contains(lab)]
    nrows = -(-len(df) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, b in zip(axes.ravel(), df.itertuples()):
        ax.imshow(X_train_all[b.row])
        ax.imshow(rare_cams[b.row].astype(np.float32), cmap="jet", alpha=0.35, vmin=0, vmax=1)
        color = "lime" if b.paste_ok else "red"
        ax.add_patch(Rectangle((b.x0 * S, b.y0 * S), (b.x1 - b.x0) * S, (b.y1 - b.y0) * S, fill=False, color=color, lw=2))
        why = "" if b.paste_ok else (" | other labels" if not b.labels_ok else " | low prob")
        ax.set_title(f"{b.image_id[:8]} p={b.prob:.2f}{why}", fontsize=7, color="black" if b.paste_ok else "red")
    fig.suptitle(f"{lab}: {len(df)} photos, {int((df.paste_ok & (df.rare == lab)).sum())} paste sources "
                 f"(CAM of the {SPECIES[lab].lower()} classes, threshold {CAM_THRESHOLD}, margin {BOX_MARGIN})")
    plt.tight_layout()
    plt.show()

for lab in RARE:
    show_rare_boxes(lab)

### Train augmentations (albumentations)
**Every real train image** (Cat, Dog, label-less and the real rare-pet photos) goes through an `albumentations` pipeline when
`CFG["aug_common"]` is on. Each epoch shows every image once, but differently each time, so the model cannot memorise the exact pixels
(the first run reached train loss 0.0005 vs val loss 0.0095). Every transform is chosen from the EDA:
- `HorizontalFlip`: pets look the same mirrored. No vertical flip, no large rotations: photos are upright.
- `RandomResizedCrop` (keeps ≥ 70% of the area, near-square) and a small `Affine` (±5% shift, 0.9–1.1 scale, ±10°): changes the framing and
  the background composition (against the scene shortcut), but mild, because a strong crop can cut a small far-away pet (Biatrix) or one of
  two cats out of the image while its label stays 1.
- Colour (`RandomBrightnessContrast` / `HueSaturationValue`), `ToGray` (≈1% grayscale photos in train and test), blur (blurry photos).
  Mild on purpose: the EDA found no train–test shift, so these regularise rather than correct a shift.

For Moris, Motya and Biatrix, each epoch also adds `RARE_COPIES` synthetic images per paste source (box above): the pet's box is
cut out, resized by a random factor and pasted at a random spot of a random **label-less** train photo, then the whole image
gets a random flip and colour change. The pet keeps its look while the room around it changes every time, so the background
no longer predicts the label. A label-less background has no cat or dog to cover (it may show other animals, e.g. a wolf
or a bird, which are not labelled), so the synthetic image gets exactly the labels of the source (rare pet + species). Sources and backgrounds come only from the train rows (`is_val = False`), never from val.

Val and test images are never augmented (only the flip-TTA at prediction time).

In [ ]:
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")   # no online version check at import
import albumentations as A
import cv2

# OpenCV (used by albumentations) starts its own thread pool: in the forked DataLoader workers that oversubscribes the 4 CPUs
# and can deadlock, so keep OpenCV single-threaded (the workers already run in parallel)
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
print("albumentations", A.__version__, "| OpenCV", cv2.__version__)

def random_resized_crop(size, **kw):
    """RandomResizedCrop across albumentations versions (>= 1.4: size=(h, w); older: height=, width=)."""
    try:
        return A.RandomResizedCrop(size=(size, size), **kw)
    except TypeError:
        return A.RandomResizedCrop(height=size, width=size, **kw)

# Every real train image (all classes): see the markdown above for why each transform is there
common_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    random_resized_crop(S, scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-10, 10), p=0.3),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10),
    ], p=0.5),
    A.ToGray(p=0.05),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=5)], p=0.1),
])

# Synthetic rare-pet images: flip + colour change of the whole pasted image
rare_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02, p=1.0),
])

def to_chw(img):
    """HWC uint8 numpy -> CHW uint8 tensor (the GPU does the float conversion and normalisation, same as for val/test)."""
    return torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)

def train_tf(img):
    img = np.asarray(img)   # plain ndarray view of the memory-mapped image (no copy)
    return to_chw(common_aug(image=img)["image"] if CFG["aug_common"] else img)

def rare_tf(img):
    return to_chw(rare_aug(image=img)["image"])

# Rare classes: cut-out pet on a new background
RARE_COPIES = 10           # synthetic images per paste source per epoch
PASTE_SCALE = (0.7, 1.1)   # random resize of the cut-out box (capped so it fits the image)

def paste_pet(src, box, background):
    """Cut `box` (fractions) out of `src` and paste it, randomly resized, at a random spot of `background` (uint8 HWC)."""
    H, W = background.shape[:2]
    x0, y0, x1, y1 = (np.array(box) * [W, H, W, H]).round().astype(int)
    pet = Image.fromarray(np.ascontiguousarray(src[y0:y1, x0:x1]))
    scale = min(random.uniform(*PASTE_SCALE), W / pet.width, H / pet.height)
    pet = pet.resize((max(1, round(pet.width * scale)), max(1, round(pet.height * scale))), Image.BILINEAR)
    px, py = random.randint(0, W - pet.width), random.randint(0, H - pet.height)
    out = background.copy()
    out[py:py + pet.height, px:px + pet.width] = np.asarray(pet)
    return out

In [ ]:
class TrainDataset(Dataset):
    """Train rows of one fold as they are, plus RARE_COPIES synthetic rare-pet images per paste source.

    Index < len(rows): a real image. Above that: a synthetic image; the background is drawn anew on every call,
    so each epoch sees different backgrounds for the same pet.
    """

    def __init__(self, rows):
        self.rows = rows
        self.labels = torch.tensor(Y_train_all, dtype=torch.float32)
        self.backgrounds = rows[Y_train_all[rows].sum(axis=1) == 0]   # label-less photos of this fold's train rows
        self.sources = rare_boxes[rare_boxes.paste_ok & rare_boxes.row.isin(rows)].reset_index(drop=True)
        self.n_synth = RARE_COPIES * len(self.sources) if len(self.backgrounds) else 0

    def __len__(self):
        return len(self.rows) + self.n_synth

    def synthesize(self, j):
        """Synthetic image from paste source j, with the labels of the source photo."""
        s = self.sources.iloc[j]
        bg = self.backgrounds[random.randrange(len(self.backgrounds))]
        img = paste_pet(X_train_all[s.row], (s.x0, s.y0, s.x1, s.y1), X_train_all[bg])
        return rare_tf(img), self.labels[s.row]

    def __getitem__(self, i):
        if i < len(self.rows):
            r = self.rows[i]
            return train_tf(X_train_all[r]), self.labels[r]
        return self.synthesize((i - len(self.rows)) % len(self.sources))

    def epoch_labels(self):
        """0/1 labels of everything one epoch shows: real rows + RARE_COPIES copies of each source's labels."""
        return np.r_[Y_train_all[self.rows], np.repeat(Y_train_all[self.sources.row.values], RARE_COPIES, axis=0)[:self.n_synth]]


def make_train_loader(rows):
    """Shuffled batches of the train rows of one fold, with the synthetic rare-pet images mixed in."""
    return DataLoader(
        TrainDataset(rows),
        batch_size=CFG["batch_size"],
        shuffle=True,
        drop_last=True,   # drop the last partial batch (keeps BatchNorm stable)
        num_workers=CFG["num_workers"],
        pin_memory=True,
        # Workers are forked ONCE and reused every epoch: forking a process that already runs CUDA, DataParallel threads
        # and W&B at the start of every epoch is a classic source of rare deadlocks. A batch that takes longer than
        # loader_timeout raises an error (with the worker's traceback) instead of waiting forever.
        persistent_workers=CFG["num_workers"] > 0,
        timeout=CFG["loader_timeout"] if CFG["num_workers"] > 0 else 0,
    )

In [ ]:
# What the augmentation produces: per rare class, two source photos (box in green) and synthetic images made from each
N_SYNTH_SHOW = 6
example_ds = TrainDataset(np.where(train.fold != FOLDS[0])[0])

fig, axes = plt.subplots(2 * len(RARE), 1 + N_SYNTH_SHOW, figsize=(2.6 * (1 + N_SYNTH_SHOW), 2.6 * 2 * len(RARE)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
ax_rows = iter(axes)
for lab in RARE:
    for j in example_ds.sources.index[example_ds.sources.rare == lab][:2]:
        ax_row = next(ax_rows)
        s = example_ds.sources.iloc[j]
        ax_row[0].imshow(X_train_all[s.row])
        ax_row[0].add_patch(Rectangle((s.x0 * S, s.y0 * S), (s.x1 - s.x0) * S, (s.y1 - s.y0) * S, fill=False, color="lime", lw=2))
        ax_row[0].set_title(f"source {s.image_id[:8]}\n{label_names(train.iloc[s.row], LABELS)}", fontsize=8)
        for ax in ax_row[1:]:
            img, y = example_ds.synthesize(j)
            ax.imshow(img.permute(1, 2, 0).numpy())
            ax.set_title("synthetic | " + ", ".join(l for l, v in zip(LABELS, y) if v) , fontsize=8)
fig.suptitle(f"Rare-pet cut-outs on label-less backgrounds (fold {FOLDS[0]} train rows)")
plt.tight_layout()
plt.show()

In [ ]:
# What the common augmentation does: 4 train images (one per class group), the original and 6 augmented versions each
rng = np.random.RandomState(SEED)
picks = [rng.choice(np.where((train.fold != 0) & (train[c] == 1))[0]) for c in ["Cat", "Dog"]]
picks += [rng.choice(np.where((train.fold != 0) & (train[LABELS].sum(axis=1) == 0))[0])]
picks += [rng.choice(np.where((train.fold != 0) & (train[RARE].sum(axis=1) > 0))[0])]
fig, axes = plt.subplots(len(picks), 7, figsize=(2.6 * 7, 2.6 * len(picks)), squeeze=False)
for ax_row, r in zip(axes, picks):
    ax_row[0].imshow(X_train_all[r])
    ax_row[0].set_title(f"original | {label_names(train.iloc[r], LABELS)}", fontsize=8)
    for ax in ax_row[1:]:
        ax.imshow(train_tf(X_train_all[r]).permute(1, 2, 0).numpy())
        ax.set_title("augmented" if CFG["aug_common"] else "aug_common = False", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Common augmentation (albumentations) on real train images")
plt.tight_layout()
plt.show()

### Model: EfficientNet backbone + 5 independent sigmoid outputs
`num_classes=5` replaces the 1000-class ImageNet head with a new linear layer. Each output is its own yes/no classifier
(sigmoid + binary cross-entropy, no softmax), so a Moris photo can be `Cat = 1` and `Moris = 1` at the same time.

Every fold trains a fresh model, so model and optimizer creation are functions.

### NEW (AI-assisted, Claude): custom layer — class-specific attention pooling
**Why.** Global average pooling (GAP) averages the whole feature map, so a small or distant pet (Biatrix in the snow) is diluted by the background,
and the room can leak into every class (shortcut). Here **every class gets its own spatial attention map** over the feature map
and pools only where its attention is high, then classifies that class-specific feature.

For a feature map $X\in\mathbb{R}^{C\times H\times W}$ (after a per-position LayerNorm) and class $k$:

$$a_k(i)=\mathrm{softmax}_i\big(t_k\,(w^{att}_k\cdot x_i+b^{att}_k)\big),\qquad f_k=\sum_i a_k(i)\,x_i,\qquad z_k=w_k\cdot f_k+b_k$$

with a $1\times1$ conv $w^{att}$ (one output channel per class), a learnable temperature $t_k=e^{\tau_k}$ and a per-class linear classifier $(w_k,b_k)$.

- **Starts as the baseline:** $w^{att}=0$ and $\tau=0$ give a uniform $a_k(i)=1/HW$, i.e. exactly GAP + linear at step 0 (checked in the cell below).
  Training then sharpens the attention only where it helps, so the comparison with the `gap` head is clean.
- **Multi-label friendly:** the 5 outputs stay independent sigmoids (same BCE loss as before), each pooled with its own attention.
- **Interpretable:** $a_k$ is the "where is Moris?" map; a gallery of these maps is drawn after the results (a complement to GradCAM, not a replacement).
- Works on any timm backbone through `forward_features` (EfficientNet and ConvNeXt here). Head parameters get the head learning rate, like the original classifier.

In [ ]:
# ===== NEW (AI-assisted, Claude): class-specific attention pooling head (custom layer) =====
from timm.layers import LayerNorm2d

class ClassAttentionPool(nn.Module):
    """Feature map (B, C, H, W) -> logits (B, K): one spatial attention map per class, class-specific pooling, class-specific linear."""

    def __init__(self, in_features, num_classes, drop_rate=0.0):
        super().__init__()
        self.norm = LayerNorm2d(in_features)                           # per-position channel normalisation (ConvNeXt features are not normalised)
        self.attn = nn.Conv2d(in_features, num_classes, kernel_size=1)  # attention score of every class at every position
        nn.init.zeros_(self.attn.weight)                                # zero scores -> uniform attention at the start == global average pooling
        nn.init.zeros_(self.attn.bias)
        self.log_temp = nn.Parameter(torch.zeros(num_classes))          # learnable sharpness t_k = exp(log_temp_k) of each class's attention
        self.drop = nn.Dropout(drop_rate)
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))   # one linear classifier per class
        nn.init.trunc_normal_(self.weight, std=0.02)
        self.bias = nn.Parameter(torch.zeros(num_classes))

    def forward(self, x, return_attn=False):
        x = self.norm(x)                                                # (B, C, H, W)
        B, C, H, W = x.shape
        scores = self.attn(x).flatten(2) * self.log_temp.exp().view(1, -1, 1)   # (B, K, H*W)
        att = scores.float().softmax(dim=-1)                            # softmax over positions, in fp32; each class sums to 1
        pooled = torch.einsum("bkn,bcn->bkc", att.to(x.dtype), x.flatten(2))    # (B, K, C): class k pools the features where *its* attention is high
        logits = (self.drop(pooled) * self.weight).sum(dim=-1) + self.bias      # (B, K)
        return (logits, att.view(B, -1, H, W)) if return_attn else logits


class AttnPoolNet(nn.Module):
    """timm backbone (feature map only, its own classifier removed) + ClassAttentionPool."""

    def __init__(self, backbone, num_classes, drop_rate):
        super().__init__()
        self.backbone = backbone
        self.pool = ClassAttentionPool(backbone.num_features, num_classes, drop_rate)

    def get_classifier(self):   # build_training() puts these parameters into the (higher-LR) head group
        return self.pool

    def forward(self, x, return_attn=False):
        return self.pool(self.backbone.forward_features(x), return_attn=return_attn)


def build_attnpool_model():
    backbone = timm.create_model(CFG["model_name"], pretrained=True, num_classes=0, drop_path_rate=CFG["drop_path_rate"])
    if hasattr(getattr(backbone, "head", None), "norm"):   # ConvNeXt: its head norm is never used (we call forward_features), so drop it
        backbone.head.norm = nn.Identity()
    return AttnPoolNet(backbone, len(LABELS), CFG["drop_rate"]).to(DEVICE).to(memory_format=MEMORY_FORMAT)


# Self-check on random data (instant, CPU): at initialisation the attention is uniform, so the head must equal GAP + linear
with torch.no_grad():
    _pool = ClassAttentionPool(32, len(LABELS)).eval()
    _x = torch.randn(2, 32, 6, 6)
    _gap_linear = _pool.norm(_x).mean(dim=(2, 3)) @ _pool.weight.T + _pool.bias
    _logits, _att = _pool(_x, return_attn=True)
    assert torch.allclose(_logits, _gap_linear, atol=1e-5), "attention head != GAP + linear at init"
    assert _att.shape == (2, len(LABELS), 6, 6) and torch.allclose(_att.sum(dim=(2, 3)), torch.ones(2, len(LABELS)), atol=1e-5)
print("ClassAttentionPool self-check passed (uniform attention at init == GAP + linear)")
del _pool, _x, _gap_linear, _logits, _att
# ===== END NEW =====

In [ ]:
def build_model():
    if CFG.get("head", "gap") == "attn":   # NEW (AI-assisted, Claude): custom head, defined in the cell above
        return build_attnpool_model()
    model = timm.create_model(
        CFG["model_name"],
        pretrained=True,
        num_classes=len(LABELS),
        drop_rate=CFG["drop_rate"],
        drop_path_rate=CFG["drop_path_rate"],
    )
    return model.to(DEVICE).to(memory_format=MEMORY_FORMAT)

model = build_model()
print(f"{CFG['model_name']}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("stages:", [name for name, _ in model.named_children()])
print("new head:", model.get_classifier())
del model   # each fold builds its own

In [ ]:
def build_training(model, steps_per_epoch):
    """Loss, optimizer, LR schedule and AMP scaler for one fold."""
    criterion = nn.BCEWithLogitsLoss()   # plain BCE, every positive and negative counts the same

    # Two parameter groups: the new head starts from random weights and needs a higher LR than the pretrained body
    head_params = list(model.get_classifier().parameters())
    head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if id(p) not in head_ids]
    max_lrs = [CFG["lr"] * CFG["backbone_lr_mult"], CFG["lr"]]   # [body, head]
    optimizer = torch.optim.AdamW(
        [{"params": body_params, "name": "body"}, {"params": head_params, "name": "head"}],
        lr=CFG["lr"], weight_decay=CFG["weight_decay"],
    )

    # Warm up for the first 10% of steps (at least 2 steps, debug runs are tiny), then cosine decay to ~0
    total_steps = steps_per_epoch * CFG["epochs"]
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lrs, total_steps=total_steps, pct_start=max(0.1, 2 / total_steps),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return criterion, optimizer, scheduler, scaler

### Training data per class (before and after augmentation)
The common classes are not augmented, so their counts stay the same. The rare-class augmentation **adds** images: each epoch
shows every real train image once plus `RARE_COPIES` synthetic images per paste source. Each synthetic image carries a rare
pet and its species, so Cat and Dog grow by the same amount as their rare pets; the label-less backgrounds are reused and
not counted again. The bars count what one epoch shows per class, before (real images only) and after adding the synthetic ones.

In [ ]:
CLASSES = LABELS + ["none"]   # "none" = images without any label

def class_counts(Y):
    return np.r_[Y.sum(axis=0), (Y.sum(axis=1) == 0).sum()].astype(int)

rows = []
for k in FOLDS:
    ds = TrainDataset(np.where(train.fold != k)[0])
    before, after = class_counts(Y_train_all[ds.rows]), class_counts(ds.epoch_labels())
    val = class_counts(Y_train_all[train.fold == k])
    for j, cls in enumerate(CLASSES):
        rows.append(dict(fold=k, label=cls, before=before[j], after=after[j], val_images=val[j],
                         share_before_pct=100 * before[j] / len(ds.rows), share_after_pct=100 * after[j] / len(ds)))
train_counts = pd.DataFrame(rows).set_index(["fold", "label"]).round(2)
display(train_counts)

fig, axes = plt.subplots(1, len(FOLDS), figsize=(9 * len(FOLDS), 4.5), squeeze=False)
for ax, (k, counts) in zip(axes[0], train_counts.groupby(level="fold")):
    counts.droplevel("fold")[["before", "after", "val_images"]].plot.bar(
        ax=ax, logy=True, rot=0, color=["lightsteelblue", "steelblue", "darkorange"])
    for bars in ax.containers:
        ax.bar_label(bars, fontsize=7)
    ax.set(title=f"Fold {k}: images per class in one epoch (log scale)", xlabel="", ylabel="images")
    ax.legend(["train, before augmentation", f"train, after (+{RARE_COPIES} synthetic per paste source)", "val images"], fontsize=8)
plt.tight_layout()
plt.show()

## 4. Training with logging
Logged every epoch: train/val loss, val mAP (threshold-free), val macro F1 (at 0.5 and with tuned thresholds, both with the species rule),
**AP, F1 and the tuned threshold per class**, both learning rates, speed and free RAM.

**Best checkpoint and early stopping** use `CFG["monitor"]`, by default the **val macro mAP**: every class counts 1/5 like the competition's
macro F1, it needs no threshold, and it is smooth. Macro F1 itself is logged every epoch (with thresholds tuned on val + species rule,
i.e. the way the submission is scored), but with 8–11 val positives per rare class one image moves a class F1 by ~5–10 points, so picking
the epoch on it would partly pick on luck. Any rise of the monitored metric saves a new checkpoint; only a rise by more than `min_delta`
resets the patience (10 epochs). `monitor="val_macroF1_tuned"` switches to macro F1 (ties then go to the better val mAP).

Every `grad_log_every` steps the **gradient norm of each network stage** is recorded for the gradient-flow plot. Training uses one GPU by
default (`multi_gpu=False`; DataParallel on Kaggle T4 x2 was slow and leaked RAM). Everything also goes to **Weights & Biases** live:
per step `train/*`, `grad_norm/*` and `mem/*`; per epoch `val/*` (`val/loss`, `val/mAP`, `val/macroF1_tuned`, `val/macroF1_t05`,
`val/AP_<class>`, `val/F1_<class>`, `val/thr_<class>`) and `epoch/*` (learning rates, timing, RAM, GPU memory). One run, kept open until the
submission is written, so the final val scores and thresholds land in the same run.

In [ ]:
@torch.no_grad()
def predict_logits(model, X, bs=None, flip=False):
    """Raw model outputs for uint8 NHWC images (numpy or GPU tensor), in batches. flip=True: mirrored images.
    On a GPU out-of-memory error the predict batch is halved and the same batch retried, so prediction never forces
    the whole training to restart."""
    bs = bs or CFG["pred_batch_size"]
    model.eval()
    out, i = [], 0
    while i < len(X):
        oom = False
        try:
            x = normalize(prep(X[i:i + bs]))
            if flip:
                x = torch.flip(x, dims=[3])
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                out.append(model(x).float().cpu())
            i += bs
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:   # outside the except block, so the failed batch's memory is really released
            x = None
            torch.cuda.empty_cache()
            if bs <= 8:
                raise RuntimeError("out of GPU memory while predicting, even with 8 images per batch")
            bs = max(8, bs // 2)
            CFG["pred_batch_size"] = bs
            print(f"!! out of GPU memory while predicting -> predict batch size {bs}")
    return torch.cat(out)

def predict_proba(model, X, tta=False):
    """Probabilities; tta=True averages the image and its horizontal flip."""
    p = torch.sigmoid(predict_logits(model, X))
    if tta:
        p = (p + torch.sigmoid(predict_logits(model, X, flip=True))) / 2
    return p.numpy()


def stage_of(param_name):
    """Group parameters by network stage: 'blocks.3.1.conv_pw.weight' -> 'blocks.3', 'conv_stem.weight' -> 'conv_stem'."""
    param_name = param_name.removeprefix("backbone.")   # NEW (AI-assisted, Claude): the attention-pool wrapper keeps the timm model in .backbone
    parts = param_name.split(".")
    if parts[0] in ("blocks", "stages"):                 # NEW (AI-assisted, Claude): 'stages.2.blocks.1...' (ConvNeXt) -> 'stages.2'
        return ".".join(parts[:2])
    return parts[0]

STAGES = list(dict.fromkeys(stage_of(n) for n, _ in build_model().named_parameters()))   # in forward order

def grad_norms_by_stage(model):
    """L2 norm of the gradient of each stage (call after the gradients are unscaled)."""
    sq = {s: torch.zeros((), device=DEVICE) for s in STAGES}
    for name, p in model.named_parameters():
        if p.grad is not None:
            sq[stage_of(name)] += p.grad.float().pow(2).sum()
    norms = torch.stack(list(sq.values())).sqrt().tolist()   # one GPU -> CPU copy for all stages
    return dict(zip(STAGES, norms))


def enforce_hierarchy(pred):
    """0/1 predictions (N, 5): an identity label switches on its species label."""
    pred = pred.copy()
    col = {lab: j for j, lab in enumerate(LABELS)}
    pred[:, col["Cat"]] |= pred[:, col["Moris"]] | pred[:, col["Motya"]]
    pred[:, col["Dog"]] |= pred[:, col["Biatrix"]]
    return pred

def macro_f1(y, pred):
    return f1_score(y, pred, average="macro", zero_division=0)

GRID = np.linspace(0.01, 0.99, 99)

def plateau_threshold(f1s):
    """Middle of the longest run of thresholds that reach the best F1."""
    idx = np.flatnonzero(f1s >= f1s.max() - 1e-9)
    runs = np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)
    run_ = max(runs, key=len)
    return float(GRID[run_[len(run_) // 2]])

def tune_thresholds(y, p):
    """Rare classes first, then Cat / Dog with the species rule applied. Returns thresholds (5,) and F1 curves."""
    col = {lab: j for j, lab in enumerate(LABELS)}
    th, curves = np.full(len(LABELS), 0.5), {}
    for lab in RARE:
        j = col[lab]
        curves[lab] = np.array([f1_score(y[:, j], (p[:, j] > t).astype(int), zero_division=0) for t in GRID])
        th[j] = plateau_threshold(curves[lab])
    rare_pred = {lab: (p[:, col[lab]] > th[col[lab]]).astype(int) for lab in RARE}
    for species, pets in [("Cat", ["Moris", "Motya"]), ("Dog", ["Biatrix"])]:
        j = col[species]
        forced = np.max([rare_pred[pet] for pet in pets], axis=0)
        curves[species] = np.array([f1_score(y[:, j], (p[:, j] > t).astype(int) | forced, zero_division=0) for t in GRID])
        th[j] = plateau_threshold(curves[species])
    return th, curves


def evaluate(model, X, y, criterion):
    """Val loss, mAP, macro F1 (at 0.5, and with thresholds tuned on these predictions; both with the species rule),
    and AP / F1 / threshold per class. The tuned macro F1 is how the submission is scored (thresholds fitted on val)."""
    logits = predict_logits(model, X)
    loss = criterion(logits.to(DEVICE), torch.tensor(y, dtype=torch.float32, device=DEVICE)).item()
    p = torch.sigmoid(logits).numpy()
    ap = {lab: average_precision_score(y[:, j], p[:, j]) for j, lab in enumerate(LABELS) if y[:, j].sum() > 0}
    th, _ = tune_thresholds(y, p)
    pred_tuned = enforce_hierarchy((p > th).astype(int))
    f1 = {lab: f1_score(y[:, j], pred_tuned[:, j], zero_division=0) for j, lab in enumerate(LABELS)}
    return dict(
        val_loss=loss,
        val_mAP=np.mean(list(ap.values())),
        val_macroF1_t05=macro_f1(y, enforce_hierarchy((p > 0.5).astype(int))),
        val_macroF1_tuned=float(np.mean(list(f1.values()))),
        **{f"AP_{lab}": v for lab, v in ap.items()},
        **{f"F1_{lab}": v for lab, v in f1.items()},
        **{f"thr_{lab}": float(t) for lab, t in zip(LABELS, th)},
    )


In [ ]:
# Host-RAM monitor and guard. A full RAM does not crash the run: the machine thrashes (keeps evicting and re-reading memory),
# the GPUs wait, and training "runs" without progress. So: measure RAM by category during training, free it when it gets low.
import gc, ctypes
try:
    import psutil
except ImportError:
    psutil = None

def _read_int(path):
    try:
        v = open(path).read().strip()
        return None if v == "max" else int(v)
    except Exception:
        return None

def _read_stat(path):
    try:
        return {k: int(v) for k, v in (line.split()[:2] for line in open(path) if len(line.split()) >= 2)}
    except Exception:
        return {}

def ram_status():
    """Host RAM in GB. available = what can still be used; anon = process memory; file = page cache (reclaimable);
    shmem = shared memory (/dev/shm, DataLoader batches); main / workers = private memory of this process / its children."""
    GB = 1e9
    limit = used = None
    anon = file = shmem = 0
    if os.path.exists("/sys/fs/cgroup/memory.current"):                      # cgroup v2 (the container limit)
        limit, used = _read_int("/sys/fs/cgroup/memory.max"), _read_int("/sys/fs/cgroup/memory.current")
        st = _read_stat("/sys/fs/cgroup/memory.stat")
        anon, file, shmem = st.get("anon", 0), st.get("file", 0), st.get("shmem", 0)
    elif os.path.exists("/sys/fs/cgroup/memory/memory.usage_in_bytes"):     # cgroup v1
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
        st = _read_stat("/sys/fs/cgroup/memory/memory.stat")
        anon, file, shmem = st.get("total_rss", st.get("rss", 0)), st.get("total_cache", st.get("cache", 0)), st.get("total_shmem", st.get("shmem", 0))
    if limit is not None and limit > 1 << 50:   # "no limit"
        limit = None
    avail = []
    if limit is not None and used is not None:
        avail.append(limit - (used - max(file - shmem, 0)))   # page cache (except shared memory) can be reclaimed
    main_gb = workers_gb = 0.0
    n_children = 0
    meminfo = _read_stat("/proc/meminfo")   # values in kB; works without psutil
    if psutil is not None:
        avail.append(psutil.virtual_memory().available)
    elif "MemAvailable:" in meminfo:
        avail.append(meminfo["MemAvailable:"] * 1024)
    if psutil is not None:
        me = psutil.Process()
        mi = me.memory_info()
        main_gb = (mi.rss - getattr(mi, "shared", 0)) / GB
        for ch in me.children(recursive=True):
            try:
                m = ch.memory_info()
                workers_gb += (m.rss - getattr(m, "shared", 0)) / GB
                n_children += 1
            except Exception:
                pass
    return dict(available_gb=min(avail) / GB if avail else float("nan"), limit_gb=(limit or 0) / GB,
                anon_gb=anon / GB, file_cache_gb=file / GB, shmem_gb=shmem / GB,
                main_private_gb=main_gb, children_private_gb=workers_gb, n_children=n_children)

def ram_text(s):
    return (f"RAM free {s['available_gb']:.1f} GB (process {s['anon_gb']:.1f}, page cache {s['file_cache_gb']:.1f}, "
            f"shm {s['shmem_gb']:.1f} | main {s['main_private_gb']:.1f}, {s['n_children']} children {s['children_private_gb']:.1f})")

def free_ram():
    """Collect garbage, give freed heap back to the OS and drop the image files from the page cache."""
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)   # glibc keeps freed memory (e.g. from DataParallel's per-step threads)
    except Exception:
        pass
    drop_file_cache()

print(ram_text(ram_status()))

In [ ]:
def train_one_fold(fold):
    """Train a fresh model on all folds but `fold`. Returns logs, OOF and test probabilities of the best epoch."""
    train_rows = np.where(train.fold != fold)[0]
    val_rows = np.where(train.fold == fold)[0]
    if CFG["debug"]:   # tiny subset for a quick pipeline check
        train_rows, val_rows = train_rows[:640], val_rows[:256]

    train_dl = make_train_loader(train_rows)
    steps = len(train_dl)
    Xva = torch.from_numpy(X_train_all[val_rows]).to(DEVICE)   # val images live on the GPU as uint8, no augmentation
    yv = Y_train_all[val_rows]
    print(f"--- fold {fold}: train {len(train_rows)} images + {train_dl.dataset.n_synth} synthetic rare-pet images "
          f"({steps} steps/epoch) | val {len(val_rows)} images")

    run = wandb.init(
        project=CFG["wandb_project"], entity=CFG["wandb_entity"], mode=CFG["wandb_mode"],
        group=CFG["model_name"].split(".")[0], name=RUN_NAME, job_type="debug" if CFG["debug"] else "train",
        config={**CFG, "validation": "scene-grouped hold-out (is_val)", "rare_copies": RARE_COPIES, "paste_scale": PASTE_SCALE,
                "n_train": len(train_rows), "n_synth": train_dl.dataset.n_synth, "n_val": len(val_rows)},
        reinit="finish_previous",   # a fold that crashed is closed when the next run starts
    )
    # Two x-axes: training steps for the per-step curves, epochs for everything printed once per epoch
    run.define_metric("train/step")
    run.define_metric("train/*", step_metric="train/step")
    run.define_metric("grad_norm/*", step_metric="train/step")
    run.define_metric("epoch")
    run.define_metric("val/*", step_metric="epoch")
    run.define_metric("epoch/*", step_metric="epoch")
    run.define_metric("mem/*", step_metric="train/step")
    run.define_metric("val/mAP", summary="max")
    run.define_metric("val/macroF1_tuned", summary="max")
    print("W&B run:", run.url if CFG["wandb_mode"] == "online" else CFG["wandb_mode"])

    net = build_model()                                            # the plain model: parameters, checkpoint, gradient norms
    model = nn.DataParallel(net) if N_GPUS > 1 else net            # what runs the batches (split over all GPUs)
    criterion, optimizer, scheduler, scaler = build_training(net, steps)
    ckpt_path = OUT_DIR / f"{RUN_NAME}_fold{fold}.pth"
    history, grad_log = [], []
    mon = CFG["monitor"]
    best_key, best_score, patience_score, epochs_without_improvement, step = (-np.inf, -np.inf), -np.inf, -np.inf, 0, 0

    stopped_for_ram = False
    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        t0 = time.time()
        run_loss = torch.zeros((), device=DEVICE)   # accumulate on the GPU: no per-step .item() sync
        done, low_ram = 0, False

        for x, y in train_dl:
            x = normalize(x.to(DEVICE, non_blocking=True).float().div_(255))   # augmented uint8 NCHW -> normalised float
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()

            if step % CFG["grad_log_every"] == 0:
                scaler.unscale_(optimizer)              # real gradient sizes, not the AMP-scaled ones
                norms = grad_norms_by_stage(net)
                grad_log.append(dict(fold=fold, step=step, epoch=epoch, **norms))
                run.log({"train/step": step, "train/loss": loss.item(),
                         **{f"train/lr_{g['name']}": g["lr"] for g in optimizer.param_groups},
                         **{f"grad_norm/{s}": v for s, v in norms.items()}})

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            run_loss += loss.detach()
            step += 1
            done += 1
            if done % CFG["log_every"] == 0:   # heartbeat: progress, speed and free RAM
                el = time.time() - t0
                ram = ram_status()
                run.log({"train/step": step, **{f"mem/{k}": v for k, v in ram.items()}})
                print(f"  epoch {epoch} step {done}/{steps} | {el:.0f}s | {done * CFG['batch_size'] / el:.0f} img/s | "
                      f"{time.strftime('%H:%M:%S')} | {ram_text(ram)}", flush=True)
                if ram["available_gb"] < CFG["ram_reserve_gb"]:
                    free_ram()
                    ram = ram_status()
                    print(f"  !! low RAM -> freed memory: {ram_text(ram)}", flush=True)
                    if ram["available_gb"] < CFG["ram_reserve_gb"]:
                        low_ram = True   # end this epoch now; the data workers are restarted below
                        break

        train_time = time.time() - t0
        t_val = time.time()
        metrics = evaluate(model, Xva, yv, criterion)
        val_time = time.time() - t_val
        lrs = {f"lr_{g['name']}": g["lr"] for g in optimizer.param_groups}
        # Best checkpoint: any rise of the monitored metric saves the weights. Macro F1 moves in steps of single val images,
        # so equal values are frequent: then the better val mAP (finer, threshold-free ranking quality) decides
        key = (metrics[mon], metrics["val_mAP"])
        improved = key > best_key
        if improved:
            best_key, best_score = key, metrics[mon]
            torch.save(net.state_dict(), ckpt_path)
        # Early stopping: only a rise of the monitored metric larger than min_delta resets the patience
        if metrics[mon] > patience_score + CFG["min_delta"]:
            patience_score, epochs_without_improvement = metrics[mon], 0
        else:
            epochs_without_improvement += 1

        ram = ram_status()
        gpu_peak = {f"gpu{d}_peak_gb": torch.cuda.max_memory_reserved(d) / 1e9 for d in range(max(N_GPUS, 1))} if DEVICE == "cuda" else {}
        history.append(dict(fold=fold, epoch=epoch, train_loss=run_loss.item() / max(done, 1), **metrics, **lrs,
                            img_per_s=done * CFG["batch_size"] / train_time, is_best=improved,
                            ram_free_gb=ram["available_gb"], **gpu_peak))
        h = history[-1]
        # Everything the printout below shows (and a bit more), once per epoch
        run.log({"epoch": epoch,
                 **{f"val/{k.removeprefix('val_')}": v for k, v in metrics.items()},   # val/loss, val/mAP, val/macroF1_t05, val/AP_<class>
                 "epoch/train_loss": h["train_loss"],
                 **{f"epoch/{k}": v for k, v in lrs.items()},                            # epoch/lr_head, epoch/lr_body
                 "epoch/train_time_s": train_time, "epoch/val_time_s": val_time,
                 "epoch/img_per_s": h["img_per_s"], "epoch/batch_size": CFG["batch_size"],
                 f"epoch/best_{mon}": best_score, "epoch/is_best": int(improved),
                 "epoch/epochs_without_improvement": epochs_without_improvement,
                 "epoch/ram_free_gb": ram["available_gb"], **{f"epoch/{k}": v for k, v in gpu_peak.items()}})
        print(f"fold {fold} epoch {epoch:2d} | train loss {h['train_loss']:.4f} | val loss {h['val_loss']:.4f} | "
              f"mAP {h['val_mAP']:.4f} | macro F1 tuned {h['val_macroF1_tuned']:.4f} (@0.5 {h['val_macroF1_t05']:.4f}) | "
              f"F1 Moris/Motya/Biatrix {h['F1_Moris']:.2f}/{h['F1_Motya']:.2f}/{h['F1_Biatrix']:.2f} | "
              f"lr head {h['lr_head']:.1e} | {train_time:.0f}s ({h['img_per_s']:.0f} img/s)"
              + ("  * best" if improved else "")
              + (f"  (no improvement {epochs_without_improvement}/{CFG['patience']})" if epochs_without_improvement else "")
              + f" | RAM free {ram['available_gb']:.1f} GB" + (" | epoch cut short: low RAM" if low_ram else ""))

        # RAM guard between epochs: free memory; if it is still low, restart the data workers (their memory is released
        # when they exit); if even that does not help, stop training here and use the best checkpoint
        free_ram()
        if low_ram or ram_status()["available_gb"] < CFG["ram_reserve_gb"]:
            train_dl = None
            free_ram()
            train_dl = make_train_loader(train_rows)
            ram = ram_status()
            print(f"  restarted the data workers: {ram_text(ram)}", flush=True)
            if ram["available_gb"] < CFG["ram_reserve_gb"]:
                print(f"!! stopping training after epoch {epoch}: only {ram['available_gb']:.1f} GB RAM free "
                      f"(< ram_reserve_gb = {CFG['ram_reserve_gb']}). The best checkpoint is used for the predictions.")
                stopped_for_ram = True
                break

        if epochs_without_improvement >= CFG["patience"]:
            print(f"early stop after epoch {epoch}: {mon} did not rise by more than {CFG['min_delta']} "
                  f"for {CFG['patience']} epochs (best {best_score:.4f})")
            break

    # Predict with the best epoch, not the last one (flip-TTA if CFG["tta"]: val and test get the same treatment)
    net.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    result = dict(
        history=pd.DataFrame(history),
        grads=pd.DataFrame(grad_log),
        val_rows=val_rows,
        oof=predict_proba(model, Xva, tta=CFG["tta"]),
        test=predict_proba(model, X_test, tta=CFG["tta"]),   # X_test: numpy uint8, streamed to the GPU in chunks
        run=run,   # kept open: the final val scores, thresholds and submission stats are logged to it at the end
    )
    run.summary.update({f"best_{mon}": best_score, "best_val_mAP_tiebreak": best_key[1], "epochs_trained": epoch,
                        "stopped_for_low_ram": stopped_for_ram})
    del model, net, Xva
    torch.cuda.empty_cache()
    return result

In [ ]:
import gc

# Train; on a CUDA out-of-memory error restart with a 25% smaller batch (multiple of 8, at least 8) and a smaller predict batch.
# The cleanup runs outside the except block, so the failed attempt's tensors are really freed first.
result = None
while result is None:
    oom = False
    try:
        result = train_one_fold(0)
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        if wandb.run is not None:
            wandb.run.finish(exit_code=1)   # mark the failed attempt in W&B
        gc.collect()
        torch.cuda.empty_cache()
        new_bs = max(8, int(CFG["batch_size"] * 0.75) // 8 * 8)
        if new_bs == CFG["batch_size"]:
            raise RuntimeError(f"out of GPU memory even with batch size {new_bs}: lower img_size or use a smaller model")
        print(f"\n!! CUDA out of memory with batch size {CFG['batch_size']} -> restarting with {new_bs}\n")
        CFG["batch_size"] = new_bs
        CFG["pred_batch_size"] = max(32, CFG["pred_batch_size"] // 2)
run = result["run"]
val_rows = result["val_rows"]
oof = np.full((len(train), len(LABELS)), np.nan)   # val probabilities in train row order (NaN = train-side row)
oof[val_rows] = result["oof"]
test_fold_probs = [result["test"]]
hist, grads = result["history"], result["grads"]
hist.to_csv(OUT_DIR / f"{RUN_NAME}_history.csv", index=False)
grads.to_csv(OUT_DIR / f"{RUN_NAME}_grad_norms.csv", index=False)
np.save(OUT_DIR / f"val_probs_{RUN_NAME}.npy", result["oof"])   # val rows only, same order as train[train.is_val]: reuse for ensembling

best = hist[hist.is_best].tail(1)[["epoch", "val_loss", "val_mAP", "val_macroF1_tuned", "val_macroF1_t05",
                                    *[f"F1_{lab}" for lab in LABELS]]].assign(epochs_trained=hist.epoch.max())
display(best.round(4))

### Training curves
The star marks the saved (best) epoch. Train loss is the mean over the epoch with augmentation and dropout on, so it can sit above the val loss early on. A val loss that rises while the train loss keeps falling means overfitting. With only ~8–11 val positives per rare class, the rare-class AP curves are noisy.

In [ ]:
panels = [
    ("train_loss", "Train loss (BCE)", False),
    ("val_loss", "Val loss (BCE)", False),
    ("lr_head", "Learning rate of the head (end of epoch)", True),
    ("val_mAP", "Val mAP (monitored)" if CFG["monitor"] == "val_mAP" else "Val mAP", False),
    ("val_macroF1_tuned", "Val macro F1, tuned thresholds (monitored)" if CFG["monitor"] == "val_macroF1_tuned" else "Val macro F1, tuned thresholds", False),
    ("val_macroF1_t05", "Val macro F1 @ 0.5", False),
]
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, (col, title, logy) in zip(axes.ravel(), panels):
    for fold, h in hist.groupby("fold"):
        line, = ax.plot(h.epoch, h[col], marker="o", label="val" if col.startswith("val") else col)
        b = h[h.is_best].iloc[-1]   # the saved checkpoint
        ax.plot(b.epoch, b[col], marker="*", ms=14, color=line.get_color(), ls="none", label="saved checkpoint")
    ax.set(title=title, xlabel="epoch", yscale="log" if logy else "linear")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

# Last two panels: AP and F1 (tuned thresholds) per class. Cat and Dog saturate early; the identity classes are where the model still learns
for ax, prefix, name in [(axes[1, 2], "AP_", "Val AP per class"), (axes[1, 3], "F1_", "Val F1 per class (tuned thresholds)")]:
    for col in [c for c in hist.columns if c.startswith(prefix)]:
        ax.plot(hist.epoch, hist[col], marker="o", label=col.removeprefix(prefix))
    ax.set(title=name, xlabel="epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

### Gradient flow
Gradient norm of each stage, from the input side (`conv_stem`) to the output side (`classifier`).
- **Healthy:** every stage gets a non-zero gradient that shrinks smoothly as the LR schedule decays.
- **Vanishing gradients:** early stages near zero while the head is large, so the pretrained body hardly adapts.
- **Exploding gradients:** spikes by orders of magnitude, often together with loss spikes.

The very first step can be missing for some stages: AMP starts with a large loss scale, the gradients overflow, and that step is skipped.

In [ ]:
g0 = grads[grads.fold == FOLDS[0]]
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))

# Left: each stage over training steps
colors = plt.cm.viridis(np.linspace(0, 1, len(STAGES)))
for stage, c in zip(STAGES, colors):
    axes[0].plot(g0.step, g0[stage], color=c, label=stage)
axes[0].set(yscale="log", xlabel="step", ylabel="gradient L2 norm", title="Gradient norm per stage during training")
axes[0].legend(fontsize=7, ncol=2)

# Right: mean per stage for each epoch (input -> output)
per_epoch = g0.groupby("epoch")[STAGES].mean()
for epoch, row in per_epoch.iterrows():
    axes[1].plot(range(len(STAGES)), row.values, marker="o", label=f"epoch {epoch}")
axes[1].set_xticks(range(len(STAGES)), STAGES, rotation=45, ha="right")
axes[1].set(yscale="log", ylabel="mean gradient L2 norm", title="Gradient flow (input -> output)")
axes[1].legend(fontsize=7)

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Thresholds on val (rare classes first, then the species rule, then Cat / Dog)
1. **Moris, Motya, Biatrix:** for each, the threshold with the best F1 on val.
2. **Species rule** with those rare predictions: Moris/Motya → Cat = 1, Biatrix → Dog = 1.
3. **Cat, Dog:** thresholds tuned **with the rule already applied**, i.e. on what will actually be submitted.

With ~8–11 positives per rare class the F1 curve is often **flat** over a range of thresholds; the **middle of the best plateau** is taken
(more robust on test than its first or last point). The val is scene-grouped: rare pets appear only in unseen scenes, where the model is
less sure, so the rare thresholds come out a bit low for test (which contains sibling shots) → slightly more rare positives on test.
The tuned val macro F1 is **optimistic** (thresholds fitted on the same images); F1 @ 0.5 and AP are not tuned and fair for comparing runs.

In [ ]:
# enforce_hierarchy, macro_f1, tune_thresholds: defined in section 4 (also used every epoch for the monitored macro F1)
has_oof = ~np.isnan(oof).any(axis=1)   # the val rows
y_oof, p_oof, oof_fold = Y_train_all[has_oof], oof[has_oof], train.fold.values[has_oof]
thresholds, f1_curves = tune_thresholds(y_oof, p_oof)
oof_pred = enforce_hierarchy((p_oof > thresholds).astype(int))

res = pd.DataFrame({
    "val_positives": y_oof.sum(axis=0),
    "AP": [average_precision_score(y_oof[:, j], p_oof[:, j]) for j in range(len(LABELS))],
    "F1_at_0_5": [f1_score(y_oof[:, j], enforce_hierarchy((p_oof > 0.5).astype(int))[:, j], zero_division=0) for j in range(len(LABELS))],
    "threshold": thresholds,
    "F1_tuned": [f1_score(y_oof[:, j], oof_pred[:, j], zero_division=0) for j in range(len(LABELS))],
}, index=LABELS).round(3)
display(res)

# Bootstrap: how much the val macro F1 moves if the val set were drawn again (thresholds fixed)
rng = np.random.RandomState(SEED)
boot = [macro_f1(y_oof[b], oof_pred[b]) for b in (rng.randint(0, len(y_oof), len(y_oof)) for _ in range(1000))]
lo, hi = np.percentile(boot, [2.5, 97.5])
val_scores = dict(val_mAP=res.AP.mean(), val_macroF1_t05=res.F1_at_0_5.mean(), val_macroF1_tuned=macro_f1(y_oof, oof_pred),
                  val_macroF1_ci_lo=lo, val_macroF1_ci_hi=hi)
print(f"val macro F1 (tuned thresholds + species rule): {val_scores['val_macroF1_tuned']:.4f}  [95% bootstrap {lo:.3f}-{hi:.3f}]  (optimistic)")
print(f"val macro F1 @ 0.5 + species rule: {val_scores['val_macroF1_t05']:.4f} | val macro AP: {val_scores['val_mAP']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for j, lab in enumerate(LABELS):
    prec, rec, _ = precision_recall_curve(y_oof[:, j], p_oof[:, j])
    axes[0].plot(rec, prec, label=f"{lab} (AP {res.AP[lab]:.2f})")
    line, = axes[1].plot(GRID, f1_curves[lab], label=lab)
    axes[1].axvline(thresholds[j], color=line.get_color(), ls=":", lw=1)
axes[0].set(xlabel="recall", ylabel="precision", title="Precision-recall curves (val)")
axes[1].set(xlabel="threshold", ylabel="F1", title="F1 vs threshold (val; Cat/Dog with the species rule; dotted = chosen)")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Results per class
TP/FP/FN are counts of val images: a **false negative (FN)** is a missed positive, a **false positive (FP)** a wrong "yes".

In [ ]:
rows = []
for k in FOLDS:
    in_k = oof_fold == k
    for j, lab in enumerate(LABELS):
        y, pred, p = y_oof[in_k, j], oof_pred[in_k, j], p_oof[in_k, j]
        rows.append(dict(
            fold=k,
            label=lab,
            val_positives=int(y.sum()),
            TP=int(((y == 1) & (pred == 1)).sum()),
            FP=int(((y == 0) & (pred == 1)).sum()),
            FN=int(((y == 1) & (pred == 0)).sum()),
            precision=precision_score(y, pred, zero_division=0),
            recall=recall_score(y, pred, zero_division=0),
            F1=f1_score(y, pred, zero_division=0),
            AP=average_precision_score(y, p) if y.sum() > 0 else np.nan,
        ))
fold_results = pd.DataFrame(rows).set_index(["fold", "label"]).round(3)
display(fold_results)

# Compact view: F1 per class (rows) and fold (columns)
f1_table = fold_results.F1.unstack("fold")
f1_table["mean"], f1_table["std"] = f1_table.mean(axis=1), f1_table.std(axis=1, ddof=0)
display(f1_table.round(3))

### Misclassified images per class
For each class: the most confident mistakes on val, one row of FN and one row of FP (a row without errors is skipped).
- **FN** (missed positives) are sorted by the lowest predicted probability first.
- **FP** (wrong "yes") are sorted by the highest predicted probability first.

The title shows the fold, error type, predicted probability for the class and the image's true labels.
Repeated patterns across folds (e.g. the same lookalike cat as a Moris FP) point at systematic errors, single odd images often at label noise.

In [ ]:
N_SHOW = 8   # images per row: the most confident mistakes of one fold and error type
oof_images = train[has_oof].reset_index(drop=True)   # same row order as y_oof / p_oof

for j, lab in enumerate(LABELS):
    grid_rows = []
    for k in FOLDS:
        for kind, is_error, most_confident_first in [
            ("FN", (y_oof[:, j] == 1) & (oof_pred[:, j] == 0), True),    # lowest probability first
            ("FP", (y_oof[:, j] == 0) & (oof_pred[:, j] == 1), False),   # highest probability first
        ]:
            m = (oof_fold == k) & is_error
            errors = oof_images[m].assign(fold=k, kind=kind, p=p_oof[m, j])
            grid_rows.append(errors.sort_values("p", ascending=most_confident_first).head(N_SHOW))

    n_errors = fold_results.xs(lab, level="label")[["FN", "FP"]].sum()
    if n_errors.sum() == 0:
        print(f"{lab}: no misclassified images")
        continue
    show_grid(
        [r for r in grid_rows if len(r)],   # a fold without FN or FP gets no row
        title=lambda r: f"fold {r.fold} {r.kind} p={r.p:.2f}\n{label_names(r, LABELS)}",
        suptitle=f"{lab}: {n_errors.FN} FN, {n_errors.FP} FP in total (rows: fold 0 FN, fold 0 FP, fold 1 FN, ...; empty rows skipped)",
        ncols=N_SHOW,
    )

### NEW (AI-assisted, Claude): attention maps of the custom head (head = "attn" only)
For each rare pet, true positives from val: the attention $a_k$ of **its own class** over the image, with the predicted probability.
A map on the animal means the pooling found it; a map on the sofa or the snow would be the shortcut. The cell does nothing for `head = "gap"`.

In [ ]:
# ===== NEW (AI-assisted, Claude): attention-map gallery =====
if CFG.get("head", "gap") == "attn":
    net_v = build_model()
    net_v.load_state_dict(torch.load(OUT_DIR / f"{RUN_NAME}_fold0.pth", map_location=DEVICE))
    net_v.eval()
    N_ATT = 6
    fig, axes = plt.subplots(len(RARE), N_ATT, figsize=(2.8 * N_ATT, 2.8 * len(RARE)), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for r, lab in enumerate(RARE):
        j = LABELS.index(lab)
        rows = val_rows[Y_train_all[val_rows, j] == 1][:N_ATT]          # val images that really show this pet
        if len(rows) == 0:
            continue
        with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits, att = net_v(normalize(prep(X_train_all[rows])), return_attn=True)
        prob = torch.sigmoid(logits.float())[:, j].cpu().numpy()
        amap = F.interpolate(att[:, j:j + 1].float(), size=(S, S), mode="bilinear")[:, 0].cpu().numpy()
        for c, row in enumerate(rows):
            m = amap[c]
            axes[r, c].imshow(X_train_all[row])
            axes[r, c].imshow((m - m.min()) / (m.max() - m.min() + 1e-8), cmap="jet", alpha=0.45)
            axes[r, c].set_title(f"{lab} p={prob[c]:.2f}", fontsize=8)
    fig.suptitle("Class-specific attention of the true class on val positives")
    plt.tight_layout()
    plt.show()
    del net_v
    torch.cuda.empty_cache()
# ===== END NEW =====

## 6. Predict test, sanity checks, submission
Test probabilities come from the same checkpoint with the same flip-TTA; the val thresholds and the species rule are applied unchanged.
Sanity checks before submitting: predicted positive rates vs train prevalence, rare positives predicted vs expected (train prevalence × test size),
the species rule holds, and what the blank white test image gets (no manual override: test is never relabelled).

In [ ]:
test_p = np.mean(test_fold_probs, axis=0)
test_pred = enforce_hierarchy((test_p > thresholds).astype(int))

submission = pd.DataFrame(test_pred, columns=LABELS)
submission.insert(0, "image_id", test.image_id.values)
assert list(submission.columns) == list(sample_sub.columns)
assert (submission.image_id.values == sample_sub.image_id.values).all()

# Sanity checks
prev = train[LABELS].mean()
checks = pd.DataFrame({
    "test_pred_%": (submission[LABELS].mean() * 100).round(2),
    "train_%": (prev * 100).round(2),
    "test_pred_count": submission[LABELS].sum(),
    "expected_count": (prev * len(test)).round(0).astype(int),
})
display(checks)
col = {lab: j for j, lab in enumerate(LABELS)}
assert not ((test_pred[:, col["Moris"]] | test_pred[:, col["Motya"]]) & (1 - test_pred[:, col["Cat"]])).any(), "species rule broken"
assert not (test_pred[:, col["Biatrix"]] & (1 - test_pred[:, col["Dog"]])).any(), "species rule broken"
for lab in RARE:
    n, exp = checks.test_pred_count[lab], checks.expected_count[lab]
    if not 0.33 * exp <= n <= 3 * exp:
        print(f"WARNING {lab}: {n} predicted vs ~{exp} expected -> check its threshold ({thresholds[col[lab]]:.2f})")
blank = submission[submission.image_id.str.startswith("d20a3fd2")]
print("blank white test image:", blank.drop(columns="image_id").to_dict("records"))

out_name = "submission_DEBUG_do_not_submit.csv" if CFG["debug"] else f"submission_{RUN_NAME}.csv"
submission.to_csv(OUT_DIR / out_name, index=False)
np.save(OUT_DIR / f"test_probs_{RUN_NAME}.npy", test_p)   # probabilities: reuse for ensembling (re-tune thresholds on averaged val probs)
print("saved", out_name)
display(submission.head())

### Results log
One row per run in `results_log.csv` (in the output folder) and the same numbers in the W&B run summary.
After submitting, fill in the **Public LB** score: val (no sibling shots) vs LB (with sibling shots) is the shortcut check —
LB far above val means the model leans on the scene.

In [ ]:
# NEW (AI-assisted, Claude): the `head` column (gap / attn) was added to the log row below
row = dict(
    run=RUN_NAME, time=time.strftime("%Y-%m-%d %H:%M"), model=CFG["model_name"], head=CFG.get("head", "gap"), img_size=CFG["img_size"],
    batch_size=CFG["batch_size"], aug_common=CFG["aug_common"], monitor=CFG["monitor"], patience=CFG["patience"], rare_copies=RARE_COPIES, tta=CFG["tta"], best_epoch=int(best.epoch.iloc[0]),
    epochs_trained=int(hist.epoch.max()), **{k: round(float(v), 4) for k, v in val_scores.items()},
    **{f"F1_{lab}": res.F1_tuned[lab] for lab in LABELS}, **{f"thr_{lab}": res.threshold[lab] for lab in LABELS},
    **{f"test_n_{lab}": int(checks.test_pred_count[lab]) for lab in LABELS}, submission=out_name, public_LB=np.nan,
)
log_path = OUT_DIR / "results_log.csv"
log = pd.concat([pd.read_csv(log_path), pd.DataFrame([row])], ignore_index=True) if log_path.exists() else pd.DataFrame([row])
log.to_csv(log_path, index=False)
display(log.tail(5))

run.summary.update({k: v for k, v in row.items() if isinstance(v, (int, float, np.integer, np.floating)) and not pd.isna(v)})
run.log({"val/thresholds": wandb.Table(dataframe=res.reset_index(names="label"))})
run.finish()

In [ ]:
# Clean up: the decoded-image cache (~14 GB) must not end up in the saved version's output.
# It is only deleted when it sits in the output folder; a cache in /kaggle/temp or /tmp disappears with the session anyway.
del X_train_all, X_test   # close the memory-mapped files first
if str(CACHE_DIR).startswith(str(OUT_DIR)):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print("deleted image cache", CACHE_DIR)
print("output files:", sorted(f.name for f in OUT_DIR.iterdir()))